In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint09_Paper_Trading

Purpose:
The lightweight paper-trading workflow: scan for opportunities, size
a position against the 1% risk rule, log it to the journal. Run this
top to bottom each time you want to open a paper trade; use the
closing cell separately whenever you're closing one out.

Author:
Alison

Version:
0.11
"""

In [ ]:
from alpha.config import DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.regime import calculate_regime
from alpha.scanner import scan_latest, top_opportunities
from alpha.position_sizing import calculate_position_size, calculate_position_size_rebalance_only
from alpha.trade_journal import (
    log_trade_open,
    log_trade_close,
    get_open_trades,
    get_closed_trades,
    summarize_journal,
    load_journal,
)

In [ ]:
config = DEFAULT_CONFIG

# Set this to your actual paper trading account size - NOT stored in
# Config since it's personal and changes over time, not something
# worth committing to git alongside the code.
ACCOUNT_SIZE = 10000

# One file, plain CSV, sitting next to this notebook's parent folder.
# Not committed to git by default - see the .gitignore note in the
# README if you want to change that.
JOURNAL_PATH = "../paper_trading/journal.csv"

print(f"Account size: {ACCOUNT_SIZE}, risk per trade: {config.risk_per_trade:.1%}")

In [ ]:
# =====================================================
# SCAN
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

scan = scan_latest(monthly_prices, regime, config)
shortlist = top_opportunities(scan, n=3)
display(shortlist)

In [ ]:
# =====================================================
# SIZE A POSITION
# =====================================================
# Pick one ticker from the shortlist above and fill these in by hand.
#
# Two ways to size, and they're not equivalent - pick deliberately:
#
# REBALANCE-ONLY (used below by default) - no hard stop-loss order.
# Matches how the strategies were actually backtested: run_backtest()
# holds a position until the next monthly rebalance drops it, with no
# stop-loss anywhere in the backtest logic. ASSUMED_ADVERSE_MOVE_PCT
# is a risk buffer for sizing purposes only - nothing will actually
# sell you out if the price moves this much.
#
# STOP-LOSS (see the commented-out block below) - sized against a
# real exit price. Matches how a discretionary trader usually thinks
# about risk, but note: this tests something the backtests never
# tested, since none of the strategies use a stop-loss internally.

TICKER = "AAPL"          # <- change this
DIRECTION = "long"       # <- "long" or "short"
STRATEGY = "Momentum"    # <- which strategy flagged it
ENTRY_PRICE = 190.00     # <- current/planned entry price
ASSUMED_ADVERSE_MOVE_PCT = 0.08  # <- risk buffer, not an exit order

sizing = calculate_position_size_rebalance_only(
    account_size=ACCOUNT_SIZE,
    entry_price=ENTRY_PRICE,
    assumed_adverse_move_pct=ASSUMED_ADVERSE_MOVE_PCT,
    direction=DIRECTION,
    config=config,
)
STOP_LOSS = None  # no real stop-loss order for a rebalance-only trade

# --- Alternative: stop-loss based sizing instead --------------------
# STOP_LOSS = 180.00
# sizing = calculate_position_size(
#     account_size=ACCOUNT_SIZE,
#     entry_price=ENTRY_PRICE,
#     stop_loss=STOP_LOSS,
#     direction=DIRECTION,
#     config=config,
# )

print(f"Shares to buy/short: {sizing.shares}")
print(f"Position value: {sizing.position_value:,.2f}")
print(f"% of account: {sizing.position_pct_of_account:.1%}")
print(f"Risk amount: {sizing.risk_amount:,.2f}")
if sizing.capped_by_max_position:
    print(f"Note: capped by max_position_pct ({config.max_position_pct:.0%}) - "
          f"the risk calculation alone would have sized this larger.")

In [ ]:
# =====================================================
# LOG THE TRADE
# =====================================================
# Only run this cell once you've actually decided to take the trade -
# it writes a new row to the journal every time it runs.

trade_id = log_trade_open(
    JOURNAL_PATH,
    ticker=TICKER,
    direction=DIRECTION,
    strategy=STRATEGY,
    entry_price=ENTRY_PRICE,
    shares=sizing.shares,
    account_size=ACCOUNT_SIZE,
    risk_amount=sizing.risk_amount,
    stop_loss=STOP_LOSS,  # None for a rebalance-only trade
    notes="",  # add anything worth remembering about why you took this
)

print(f"Logged. trade_id = {trade_id}")
print("Keep this trade_id somewhere - you'll need it to close the trade later.")

In [ ]:
# =====================================================
# CURRENT JOURNAL STATE
# =====================================================

print("Open trades:")
display(get_open_trades(JOURNAL_PATH))

print("\nClosed trades:")
display(get_closed_trades(JOURNAL_PATH))

print("\nSummary:")
print(summarize_journal(JOURNAL_PATH))

In [ ]:
# =====================================================
# CLOSING A TRADE (run this separately, whenever you're ready)
# =====================================================
# Paste in the trade_id you saved earlier, and the price you're
# closing at.

# trade_id_to_close = "AAPL-20260719120000"  # <- uncomment and fill in
# exit_price = 205.00                          # <- uncomment and fill in
# log_trade_close(JOURNAL_PATH, trade_id_to_close, exit_price)
# print("Closed.")

In [ ]:
# =====================================================
# NOTES
# =====================================================
# This is deliberately lightweight - not the full Sprint 10 (Portfolio
# Management). What's NOT covered here: sector exposure, cash
# allocation across multiple simultaneous positions, correlation
# between holdings, or automated stop-loss placement. If you open
# several positions at once, nothing here checks whether they're all
# in the same sector or otherwise correlated - that's still on you to
# judge, or a future sprint to automate.
#
# max_position_pct (20% default) is a blunt safety cap, not a
# diversification strategy - it stops any single position from being
# oversized, it doesn't manage the portfolio as a whole.
#
# Rebalance-only vs stop-loss: this notebook defaults to rebalance-
# only sizing because that's what the backtests in Sprint 5-6 actually
# tested - none of them use a stop-loss. If you switch to stop-loss
# sizing for paper trading, you're testing a different (possibly
# better, possibly worse) system than the one the backtest numbers
# describe. Worth being deliberate about which one you're doing, and
# consistent about it across trades so your paper trading results mean
# something.